---
title: "Persistence, Replay, and Agent Memory"
draft: true
categories: [agents, workflows, langgraph, ai-engineering, search, retrieval, memory, reliability]
---

Checkpointing turns an in-process investigation into a resumable thread. Long-term memory solves a different problem: selecting validated experience from earlier investigations and deciding whether it is still applicable to the target repository revision.


## Restart from a durable interrupt

The SQLite database lives under the repository's gitignored `.tmp` directory. A stable `thread_id` identifies the paused investigation, while the effect ledger proves that completed searches and the final export are not repeated after a restart.


In [1]:
import os
from pathlib import Path

os.environ["LANGGRAPH_STRICT_MSGPACK"] = "true"
from langgraph.checkpoint.sqlite import SqliteSaver
from langgraph.types import Command
from change_planner.fixtures import request_for
from change_planner.workflow import build_change_planner_graph, make_context

database = Path.cwd() / ".tmp" / "change-planner" / "chapter-07.sqlite"
database.parent.mkdir(parents=True, exist_ok=True)
database.unlink(missing_ok=True)
config = {"configurable": {"thread_id": "dry-run-restart"}}
context = make_context()

with SqliteSaver.from_conn_string(str(database)) as saver:
    graph = build_change_planner_graph(checkpointer=saver)
    paused = graph.invoke(
        {"request": request_for("dry-run-01").model_dump(), "events": [], "branch_results": []},
        config=config,
        context=context,
        version="v2",
    )
    print("paused at", graph.get_state(config).next)

with SqliteSaver.from_conn_string(str(database)) as saver:
    restarted = build_change_planner_graph(checkpointer=saver)
    completed = restarted.invoke(
        Command(resume={"action": "approve", "reason": "restart review"}),
        config=config,
        context=context,
        version="v2",
    )
    history = list(restarted.get_state_history(config))

print({"status": completed.value["status"], "checkpoints": len(history), "effects": context.controller.effects})
assert completed.value["status"] == "complete"
assert context.controller.effects.count("search:behavior") == 1
assert context.controller.effects.count("export:plan") == 1


paused at ('review',)
{'status': 'complete', 'checkpoints': 13, 'effects': ['search:behavior', 'search:tests', 'search:operations', 'search:history', 'test:test-link:fixture/change-cli@8f2c1d:tests/test_clear_outputs.py', 'export:plan']}


Reconstructing the saver and graph did not repeat investigation branches. The checkpoint owns workflow progress; the idempotent effect ledger independently proves which search, test, memory, and export effects occurred. A real deployment would also define access control, retention, secret exclusion, and deletion policies.

## Memory is selective and revision-aware

The next run shares an episodic memory store. The memory hit is useful prior experience, but its supporting evidence remains tied to the repository revision and must be checked before it becomes current evidence.


In [2]:
from change_planner.fixtures import request_for, snapshot
from change_planner.schemas import MemoryRecord
from change_planner.workflow import MemoryStore, run_fixture


memory = MemoryStore()
first = run_fixture("dry-run-01", memory=memory)
second = run_fixture("dry-run-01", memory=memory)
print({
    "first": first["status"],
    "second": second["status"],
    "memory_hits": len(second["memory_hits"]),
    "stored_ids": [record.id for record in memory.records],
})
assert first["status"] == second["status"] == "complete"
assert any(item["investigation_id"] == "dry-run-01" for item in second["memory_hits"])

stale = MemoryRecord(
    id="stale-example",
    kind="episodic",
    repository="fixture/change-cli",
    valid_from="old-revision",
    text="dry-run clear outputs",
    investigation_id="old-run",
    content_fingerprint="sha256:old",
    confidence=0.8,
    status="reviewed",
)
stale_memory = MemoryStore()
stale_memory.add(stale)
stale_hits = stale_memory.recall(request_for("dry-run-01"), snapshot())
print({"stale_hits": len(stale_hits), "stale_status": stale_memory.records[-1].status})
assert stale_hits == []
assert stale_memory.records[-1].status == "invalidated"


{'first': 'complete', 'second': 'complete', 'memory_hits': 1, 'stored_ids': ['fixture/change-cli:dry-run-01']}
{'stale_hits': 0, 'stale_status': 'invalidated'}


An agent memory policy needs admission, consolidation, retrieval, freshness, invalidation, and deletion rules. Storing every model sentence would turn prior guesses into future evidence. Chapter 08 measures retrieval, impact, workflow, and memory contracts separately.
